### change in available issuance-start date and end date
### top 10 across all methods

In [9]:
import re
import pandas as pd
from pathlib import Path
import numpy as np
from openpyxl.styles import Font
pd.options.display.float_format = '{:,.0f}'.format

In [2]:
# Google Sheets packages
import gspread
from gspread_dataframe import set_with_dataframe
from google.oauth2.service_account import Credentials

# Define the scopes and credentials
SCOPES = ['https://www.googleapis.com/auth/drive.file', 
          'https://www.googleapis.com/auth/drive',
          'https://www.googleapis.com/auth/spreadsheets' ]

# Google API keys
#credentials1 = Credentials.from_service_account_file('C:\\Users\\Pawandeep Kaur\\RepuTex\\Company - Documents\\10 Python\\API Keys\\vcm-database.json', scopes=SCOPES)
credentials1 = Credentials.from_service_account_file(Path.home() / "RepuTex" / "Company - Documents" / "10 Python" /"API Keys"/ "vcm-database.json", scopes=SCOPES)
# Authorize with gspread
gc = gspread.authorize(credentials1)


from gspread.utils import rowcol_to_a1

c:\Users\Pawandeep Kaur\Python\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.0) doesn't match a supported version!
  warnings.warn(


In [3]:

def format_numeric_cells_with_commas(ws, nrows, ncols, header_rows=1):
    """
    Applies comma number format to numeric-looking columns (based on row after header).
    Formats rows (header_rows+1 .. nrows), columns that look numeric.
    """
    if nrows <= header_rows or ncols < 2:
        return

    # values we just wrote (so we can detect numeric columns)
    vals = ws.get_values(f"A1:{rowcol_to_a1(nrows, ncols)}")

    # guard
    if len(vals) <= header_rows:
        return

    # use first data row to decide which columns are numeric
    sample_row = vals[header_rows]
    numeric_cols = []
    for c in range(ncols):
        v = sample_row[c] if c < len(sample_row) else ""
        if isinstance(v, (int, float)):
            numeric_cols.append(c + 1)
            continue
        if isinstance(v, str):
            s = v.replace(",", "").strip()
            if s == "":
                continue
            try:
                float(s)
                numeric_cols.append(c + 1)
            except:
                pass

    # apply format column-by-column (simple and reliable)
    start_row = header_rows + 1
    end_row = nrows
    for col in numeric_cols:
        a1 = f"{rowcol_to_a1(start_row, col)}:{rowcol_to_a1(end_row, col)}"
        ws.format(a1, {"numberFormat": {"type": "NUMBER", "pattern": "#,##0"}})


In [4]:
NBSP = "\u00A0"
def make_key(s: pd.Series) -> pd.Series:
    return (s.astype("string")
             .fillna("")
             .str.replace(NBSP, " ", regex=False)
             .str.replace(r"\s+", " ", regex=True)
             .str.strip()
             .str.upper())

def add_display_and_key(df: pd.DataFrame, col: str, key_col: str):
    df[key_col] = make_key(df[col])
    nice_map = (df.loc[df[key_col] != ""]
                  .groupby(key_col)[col]
                  .agg(lambda x: x.value_counts().index[0]))

    return df, nice_map


In [5]:
def asat_from_filename(fp):
    name = Path(fp).name

    # YYYY-MM-DD
    m = re.search(r'(\d{4}-\d{2}-\d{2})', name)
    if m:
        return pd.Timestamp(m.group(1))

    # YYYYMMDD
    m = re.search(r'(\d{8})', name)
    if m:
        return pd.to_datetime(m.group(1), format="%Y%m%d")

    # YYYY-MM (assume end of month)
    m = re.search(r'(\d{4}-\d{2})', name)
    if m:
        return pd.to_datetime(m.group(1) + "-01") + pd.offsets.MonthEnd(0)

    # no usable date -> ignore file
    return None

ERF_DIR = Path.home()/"RepuTex"/"Company - Documents"/"3 Research"/"2 Models and Databases"/"9 Registry Database"/"ERF Registry"  
CAC_DIR = Path.home()/"RepuTex"/"Company - Documents"/"3 Research"/"2 Models and Databases"/"9 Registry Database"/"CAC Registry"    

# If CSV:
erf_files = sorted(ERF_DIR.glob("*.xlsx"))
cac_files = sorted(CAC_DIR.glob("*.xlsx"))


start_date = pd.Timestamp("2025-07-01")
end_date   = pd.Timestamp("2025-12-31")

# Option B needs the "as at" snapshots at:
asat_start = start_date - pd.Timedelta(days=1)  # 2025-06-30
asat_end   = end_date                           # latest register date



In [6]:
print(ERF_DIR)

C:\Users\Pawandeep Kaur\RepuTex\Company - Documents\3 Research\2 Models and Databases\9 Registry Database\ERF Registry


In [6]:
def load_snapshot(files, target_asat, reader, min_year=2024):
    d = {}
    for f in files:
        dt = asat_from_filename(f)
        if dt is None:
            continue
        if dt.year < min_year:          
            continue
        d[dt] = f

    if not d:
        raise ValueError("No usable snapshot files after filtering.")

    available = sorted(d.keys())
    candidates = [dt for dt in available if dt <= target_asat]
    if not candidates:
        raise ValueError(f"No snapshot <= {target_asat.date()} found.")
    chosen = max(candidates)

    out = reader(d[chosen])
    out["as_at"] = chosen
    return out, chosen


In [10]:
#mapping_fp = "C:\\Users\\Pawandeep Kaur\\RepuTex\\Company - Documents\\10 Python\\Codes\\Supply Tool\\Data Files\\Method Mapping.xlsx"
mapping_fp = (Path.home() / "RepuTex" / "Company - Documents" / "10 Python" / "Codes" / "Supply Tool" / "Data Files"/"Method Mapping.xlsx")

method_map_df = pd.read_excel(mapping_fp)

method_map_df = method_map_df[['Method', 'Method Group']].copy()
method_map_df['Method Keyword'] = method_map_df['Method'].astype(str).str.strip()
method_map_df['Method Group'] = method_map_df['Method Group'].astype(str).str.strip()

method_keywords = method_map_df['Method Keyword'].tolist()
method_mapping = dict(zip(method_map_df['Method Keyword'], method_map_df['Method Group']))

In [8]:

def add_method_group(erf_df, method_keywords, method_mapping, method_col="Method"):
    s = erf_df[method_col].astype("string").fillna("")

    # find first matching keyword in the Method text
    kw = s.apply(lambda x: next((k for k in method_keywords if k in x), None))

    # map keyword -> method group
    erf_df["Method Group"] = kw.map(method_mapping).fillna("Unknown")
    
    erf_df.loc[erf_df["Project ID"].astype(str).eq("EOP100557"), "Method Group"] = "Reforestation and afforestation"

    return erf_df


In [9]:

COL_AGENT      = "Agent"
COL_PROJECT_ID = "Project ID"          
COL_PROPONENT  = "Project Proponent"
COL_METHOD_TXT = "Method"
COL_ISSUED     = "ACCUs Total units issued"
COL_STATUS     = "Status"
COL_VOL        = "Original volume of abatement committed under contract"
COL_DELIVER    = "Volume of abatement sold to the Commonwealth under contract"
COL_RELEASE    = "Volume released through Fixed Delivery exit arrangements or lapsed Optional Delivery milestones"
COL_LAPSED     = "CAC lapsed"
COL_METHOD = "Method Group"   
COL_REVOKED = "Project revoked"
'''def prep_erf(erf_df):
    # create Method Group
    erf_df = add_method_group(erf_df, method_keywords, method_mapping, method_col=COL_METHOD_TXT)

    # clean agent
    erf_df[COL_AGENT] = erf_df[COL_AGENT].astype("string").fillna("Unknown").str.strip()
    
    # numeric
    erf_df[COL_ISSUED] = pd.to_numeric(erf_df[COL_ISSUED], errors="coerce").fillna(0)

    return erf_df[[COL_PROJECT_ID, COL_PROPONENT, COL_AGENT, "Method Group", COL_ISSUED]].copy()'''
    
    
def prep_erf(erf_df):
    erf_df = add_method_group(erf_df, method_keywords, method_mapping, method_col=COL_METHOD_TXT)
    revoked_flag = (erf_df[COL_REVOKED].astype("string")
                    .fillna("")
                    .str.replace(r"\s+", " ", regex=True)
                    .str.strip()
                    .str.casefold())
    erf_df = erf_df[revoked_flag.eq("no")].copy()
    erf_df[COL_AGENT] = erf_df[COL_AGENT].astype("string").fillna("Unknown").str.strip()
    erf_df[COL_ISSUED] = pd.to_numeric(erf_df[COL_ISSUED], errors="coerce").fillna(0)

    return erf_df[[COL_PROJECT_ID, COL_PROPONENT, COL_AGENT, "Method Group", COL_ISSUED]].copy()

def prep_cac(cac_df):
    for c in [COL_VOL, COL_DELIVER, COL_RELEASE]:
        cac_df[c] = pd.to_numeric(cac_df[c], errors="coerce").fillna(0)

    # lapsed = sum(original volume) where status is lapsed/terminated
    lapsed = (cac_df.loc[cac_df[COL_STATUS].eq("Lapsed/Terminated")]
                    .groupby(COL_PROJECT_ID)[COL_VOL].sum())

    # delivered/released often already project-level
    agg = (cac_df.groupby(COL_PROJECT_ID, as_index=False)
                .agg({COL_DELIVER:"sum", COL_RELEASE:"sum"}))

    agg["CAC lapsed"] = agg[COL_PROJECT_ID].map(lapsed).fillna(0)
    return agg

def build_available(erf_snap, cac_snap):
    m = erf_snap.merge(cac_snap, on=COL_PROJECT_ID, how="left")

    for c in [COL_DELIVER, COL_RELEASE, COL_LAPSED]:
        m[c] = pd.to_numeric(m[c], errors="coerce").fillna(0)

    #m["Available issuance"] = m[COL_ISSUED] - m[COL_DELIVER] - m[COL_RELEASE] - m[COL_LAPSED]
    m["Available issuance"] = m[COL_ISSUED] - m[COL_DELIVER]
    return m

In [10]:
def read_erf(fp):
    return pd.read_excel(fp, header=3)

def read_cac(fp):
    return pd.read_excel(fp, header=2)

erf_start_raw, erf_start_dt = load_snapshot(erf_files, asat_start, read_erf)
erf_end_raw,   erf_end_dt   = load_snapshot(erf_files, asat_end,   read_erf)

cac_start_raw, cac_start_dt = load_snapshot(cac_files, asat_start, read_cac)
cac_end_raw,   cac_end_dt   = load_snapshot(cac_files, asat_end,   read_cac)



c:\Users\Pawandeep Kaur\Python\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\Pawandeep Kaur\Python\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


### Available issuance-proponents

In [11]:
start = build_available(prep_erf(erf_start_raw), prep_cac(cac_start_raw))
end   = build_available(prep_erf(erf_end_raw),   prep_cac(cac_end_raw))

COL_PROPONENT_KEY = "Project Proponent_key"

end, proponent_nice = add_display_and_key(end, "Project Proponent", "Project Proponent_key")
start, _            = add_display_and_key(start, "Project Proponent", "Project Proponent_key")


delta = (end[[COL_PROJECT_ID,COL_PROPONENT_KEY, "Project Proponent", "Method Group", "Available issuance"]]
         .merge(start[[COL_PROJECT_ID, "Available issuance"]],
                on=COL_PROJECT_ID, how="left", suffixes=("_end", "_start")))

delta["Available issuance_start"] = delta["Available issuance_start"].fillna(0)
delta["Delta Available issuance"] = delta["Available issuance_end"] - delta["Available issuance_start"]

# display name per key (most frequent original)
display_map = (
    delta.loc[delta[COL_PROPONENT].notna(), [COL_PROPONENT_KEY, COL_PROPONENT]]
        .assign(**{
            COL_PROPONENT: lambda x: (
                x[COL_PROPONENT].astype(str)
                  .str.replace(r"\s+", " ", regex=True)
                  .str.strip()
            )
        })
        .groupby(COL_PROPONENT_KEY)[COL_PROPONENT]
        .agg(lambda s: s.value_counts().index[0])
)

# top 10 by KEY
totals_delta = (delta.groupby(COL_PROPONENT_KEY, as_index=False)["Delta Available issuance"]
                .sum()
                .sort_values("Delta Available issuance", ascending=False)
                .head(15))

top10_list_delta = totals_delta[COL_PROPONENT_KEY].tolist()

method_totals_top10 = (
    delta[delta[COL_PROPONENT_KEY].isin(top10_list_delta)]
      .groupby("Method Group")["Delta Available issuance"]
      .sum()
)

all_method_groups = (
    method_totals_top10
      .reindex(delta["Method Group"].dropna().unique(), fill_value=0)  
      .sort_values(ascending=False)
      .index
      .tolist()
)

table_top10_delta = (
    delta[delta[COL_PROPONENT_KEY].isin(top10_list_delta)]
      .pivot_table(index=COL_PROPONENT_KEY,
                   columns="Method Group",
                   values="Delta Available issuance",
                   aggfunc="sum",
                   fill_value=0)
      .reindex(index=top10_list_delta, columns=all_method_groups, fill_value=0)
      .reset_index()
)

# add original display name + make it the main column
table_top10_delta["Project Proponent"] = table_top10_delta[COL_PROPONENT_KEY].map(display_map)

# put Project Proponent first, and drop the key (optional but usually desired)
first_cols = ["Project Proponent"]
other_cols = [c for c in table_top10_delta.columns if c not in first_cols + [COL_PROPONENT_KEY]]
table_top10_delta = table_top10_delta[first_cols + other_cols]

table_top10_delta.head(2)


Method Group,Project Proponent,Landfill gas,Human Induced Regeneration,Carbon capture and storage,Savanna fire management (A),Avoided deforestation,Energy efficiency,Plantation forestry,Avoided clearing,Beef cattle herd management,...,Soil carbon,Savanna fire management (S+A),Reforestation and afforestation,Native Forest from Managed Regrowth,Alternative waste treatment,Verified Carbon Standard,Wastewater,Fugitives - Coal,Transport - Land and sea,Blue Carbon
0,Terra Carbon Pty Limited,0,"1,323,551",0,"1,413","284,819",0,0,"104,691",0,...,0,0,0,0,0,0,0,0,0,0
1,LMS Energy Pty Ltd,"1,687,750",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:

totals_as_at = (end.groupby(COL_PROPONENT_KEY, as_index=False)["Available issuance"]
               .sum()
               .rename(columns={"Available issuance": "total_available_issuance"})
         )

top10_list_as_at = (totals_as_at.sort_values("total_available_issuance", ascending=False)
                    .head(15)[COL_PROPONENT_KEY]
                    .tolist()
            )
#all_method_groups = sorted(end[COL_METHOD].dropna().unique(), reverse=True)

method_totals_top10 = (
    end[end[COL_PROPONENT_KEY].isin(top10_list_as_at)]
      .groupby("Method Group")["Available issuance"]
      .sum()
)

all_method_groups = (
    method_totals_top10
      .reindex(end["Method Group"].dropna().unique(), fill_value=0)  # keep all
      .sort_values(ascending=False)
      .index
      .tolist()
)
table_across_as_at = (end[end[COL_PROPONENT_KEY].isin(top10_list_as_at)]
                .pivot_table(index=COL_PROPONENT_KEY,
                             columns=COL_METHOD,
                             values="Available issuance",
                             aggfunc="sum",
                             fill_value=0)
                .reindex(index=top10_list_as_at, columns=all_method_groups, fill_value=0)
                .reset_index()
               )

# add original display name + make it the main column
table_across_as_at["Project Proponent"] = table_across_as_at[COL_PROPONENT_KEY].map(display_map)

# put Project Proponent first, and drop the key (optional but usually desired)
first_cols = ["Project Proponent"]
other_cols = [c for c in table_across_as_at.columns if c not in first_cols + [COL_PROPONENT_KEY]]
table_across_as_at = table_across_as_at[first_cols + other_cols]


table_across_as_at



Method Group,Project Proponent,Landfill gas,Human Induced Regeneration,Savanna fire management (A),Avoided deforestation,Environmental plantings,Alternative waste treatment,Avoided clearing,Soil carbon,Manure management,...,Native Forest from Managed Regrowth,Savanna fire management (S+A),Reforestation and afforestation,Verified Carbon Standard,Wastewater,Fugitives - Coal,Beef cattle herd management,Transport - Land and sea,Carbon capture and storage,Blue Carbon
0,LMS Energy Pty Ltd,"23,588,482",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Terra Carbon Pty Limited,0,"7,790,401","449,174","3,321,581",0,0,"445,874",0,0,...,0,0,0,0,0,0,0,0,0,0
2,EDL LFG (NSW) Pty Ltd,"4,207,250",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Veolia Environmental Services (Australia) Pty Ltd,"2,688,540",0,0,0,0,"133,914",0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,LGI Limited,"2,631,108",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,ALFA (NT) Limited,0,0,"2,608,022",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,ALFA (NT) Limited; Jawoyn Association Aborigin...,0,0,"2,559,739",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,Landfill Operations Pty Ltd,"2,370,271",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,LANIN HOLDINGS PTY LTD; NINAL VENTURES PTY LTD,0,"1,890,477",0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,PANIRI HOLDINGS PTY LTD; Paniri Ventures Pty Ltd,0,"1,705,745",0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
def top10_by_method(df, value_col, proponent_col="Project Proponent", method_col="Method Group", n=15):
    # Sum at (method, proponent) level
    mp = (df.groupby([method_col, proponent_col], as_index=False)[value_col]
            .sum())

    # Rank within each method
    mp["rank"] = (mp.groupby(method_col)[value_col]
                    .rank(method="first", ascending=False)
                    .astype(int))

    # Keep top N per method
    top = mp[mp["rank"] <= n].copy()

    # Return:
    # A) combined long table
    top_long = top.sort_values([method_col, "rank"])

    # B) dictionary of pretty 2-col tables per method
    tables = {}
    for m, sub in top_long.groupby(method_col):
        tables[m] = (sub[[proponent_col, value_col]]
                     .set_index(proponent_col)
                     .sort_values(value_col, ascending=False))

    return top_long, tables


In [14]:
#by method as at
top_asat_long, top_asat_tables = top10_by_method(end, value_col="Available issuance")

In [15]:
#by method from date 2 to date 1
top_delta_long, top_delta_tables = top10_by_method(delta, value_col="Delta Available issuance")

In [16]:
'''def top10_by_method_tables(df, value_col,
                           proponent_col="Project Proponent",
                           method_col="Method Group",
                           n=15):
    mp = (df.groupby([method_col, proponent_col], as_index=False)[value_col]
            .sum()
            .sort_values([method_col, value_col], ascending=[True, False]))

    tables = {}
    for m, sub in mp.groupby(method_col):
        tbl = (sub.head(n)
               .set_index(proponent_col)[[value_col]]
               .rename(columns={value_col: str(m)}))   
        tables[m] = tbl

    return tables'''

def top10_by_method_tables(df, value_col,
                           proponent_col="Project Proponent",
                           method_col="Method Group",
                           n=15,
                           exclude_unknown=False):
    d = df.copy()
    
    if exclude_unknown:
        key = (d[proponent_col].astype("string")
                 .fillna("")
                 .str.replace(r"\s+", " ", regex=True)
                 .str.strip()
                 .str.upper())
        d = d[~key.isin(["", "UNKNOWN"])].copy()

    # 2) Sum at (method, proponent) level and sort
    mp = (d.groupby([method_col, proponent_col], as_index=False)[value_col]
            .sum()
            .sort_values([method_col, value_col], ascending=[True, False]))

    # 3) Take top N per method
    tables = {}
    for m, sub in mp.groupby(method_col):
        tbl = (sub.head(n)
               .set_index(proponent_col)[[value_col]]
               .rename(columns={value_col: str(m)}))
        tables[m] = tbl

    return tables

def write_tables_to_sheet(writer, sheet_name, tables_dict, number_format="#,##0.00"):
    # Create empty sheet first
    pd.DataFrame().to_excel(writer, sheet_name=sheet_name, index=False)
    ws = writer.book[sheet_name]

    row = 1  # 1-based for openpyxl
    for method, tbl in tables_dict.items():
        # Method title
        ws.cell(row=row, column=1, value=str(method))
        ws.cell(row=row, column=1).font = Font(bold=True)
        row += 1

        # Write table starting at current row (pandas uses 0-based startrow)
        startrow0 = row - 1
        tbl.to_excel(writer, sheet_name=sheet_name, startrow=startrow0, startcol=0)

        # Apply number format to the value column (column B) for the written rows
        nrows = tbl.shape[0]
        for r in range(row + 1, row + 1 + nrows):  # +1 because header row
            ws.cell(row=r, column=2).number_format = number_format

        # Move down: header + data + 2 blank rows
        row += (nrows + 2 + 2)

    # Optional: widen columns
    ws.column_dimensions["A"].width = 45
    ws.column_dimensions["B"].width = 22


# 1) Build separate Top-10 tables per method
tables_asat  = top10_by_method_tables(end,  value_col="Available issuance",exclude_unknown=True)
tables_delta = top10_by_method_tables(delta,   value_col="Delta Available issuance",exclude_unknown=True)



### Available issuance-Agents

In [17]:
GROUP_COL = "Agent" 
GROUP_KEY = "Agent_key" 
start_df = build_available(prep_erf(erf_start_raw), prep_cac(cac_start_raw))
end_df = build_available(prep_erf(erf_end_raw), prep_cac(cac_end_raw))

end_df, agent_nice = add_display_and_key(end_df, "Agent", "Agent_key")
start_df, _        = add_display_and_key(start_df, "Agent", "Agent_key")

end_prop_agent = end_df

totals = (end_df.groupby(GROUP_KEY, as_index=False)["Available issuance"]
               .sum()
               .rename(columns={"Available issuance": "total_available_issuance"}))

totals = totals[~totals[GROUP_KEY].isin(["", "UNKNOWN"])].copy()

top10_list = (totals.sort_values("total_available_issuance", ascending=False)
                    .head(15)[GROUP_KEY]
                    .tolist())
#all_method_groups = sorted(end[COL_METHOD].dropna().unique(), reverse=True)
# method groups ordered by DESC total across TOP10 only (but keep ALL method groups)
method_totals_top10 = (
    end_df[end_df[GROUP_KEY].isin(top10_list)]
      .groupby("Method Group")["Available issuance"]
      .sum()
)

all_method_groups = (
    method_totals_top10
      .reindex(end_df["Method Group"].dropna().unique(), fill_value=0)  
      .sort_values(ascending=False)
      .index
      .tolist()
)

table_asat_by_agent = (end_df[end_df[GROUP_KEY].isin(top10_list)]
    .pivot_table(index=GROUP_KEY, columns=COL_METHOD, values="Available issuance",
                 aggfunc="sum", fill_value=0)
    .reindex(index=top10_list, columns=all_method_groups, fill_value=0)
    .reset_index()
)

table_asat_by_agent = table_asat_by_agent.rename(columns={"Agent_key": "Agent"})

table_asat_by_agent



delta = (end_df[[COL_PROJECT_ID, GROUP_COL, GROUP_KEY, "Method Group", "Available issuance"]]
         .merge(start_df[[COL_PROJECT_ID, "Available issuance"]],
                on=COL_PROJECT_ID, how="left", suffixes=("_end", "_start")))

delta["Available issuance_start"] = delta["Available issuance_start"].fillna(0)
delta["Delta Available issuance"] = delta["Available issuance_end"] - delta["Available issuance_start"]


totals = (delta.groupby(GROUP_KEY, as_index=False)["Delta Available issuance"]
               .sum()
               .rename(columns={"Delta Available issuance": "total_delta"}))

totals = totals[~totals[GROUP_KEY].isin(["", "UNKNOWN"])].copy()

top10_list = (totals.sort_values("total_delta", ascending=False)
                    .head(15)[GROUP_KEY]
                    .tolist())
#all_method_groups = sorted(delta[COL_METHOD].dropna().unique(), reverse=True)

method_totals_top10 = (
    delta[delta[GROUP_KEY].isin(top10_list)]
      .groupby("Method Group")["Delta Available issuance"]
      .sum()
)

all_method_groups = (
    method_totals_top10
      .reindex(delta["Method Group"].dropna().unique(), fill_value=0) 
      .sort_values(ascending=False)
      .index
      .tolist()
)
table_delta_by_agent = (delta[delta[GROUP_KEY].isin(top10_list)]
    .pivot_table(index=GROUP_KEY, columns="Method Group", values="Delta Available issuance",
                 aggfunc="sum", fill_value=0)
    .reindex(index=top10_list, columns=all_method_groups, fill_value=0)
    .reset_index()
)

'''table_delta_by_agent[GROUP_COL] = table_delta_by_agent[GROUP_KEY].map(agent_nice)
table_delta_by_agent = table_delta_by_agent.drop(columns=[GROUP_KEY])
cols = [GROUP_COL] + [c for c in table_delta_by_agent.columns if c != GROUP_COL]
table_delta_by_agent = table_delta_by_agent[cols]'''
table_delta_by_agent = table_delta_by_agent.rename(columns={"Agent_key": "Agent"})

table_delta_by_agent


tables_asat_agent  = top10_by_method_tables(end_df,  value_col="Available issuance",
                                            proponent_col=GROUP_COL, method_col="Method Group",exclude_unknown=True)

tables_delta_agent = top10_by_method_tables(delta,   value_col="Delta Available issuance",
                                            proponent_col=GROUP_COL, method_col="Method Group",exclude_unknown=True)



In [18]:
out_path = "Available_issuance.xlsx"

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:    
    table_across_as_at.to_excel(writer, sheet_name="Prop as at latest reg", index=False)
with pd.ExcelWriter(out_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    table_top10_delta.to_excel(writer, sheet_name="Prop delta jun25-latest reg", index=False)
with pd.ExcelWriter(out_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    write_tables_to_sheet(writer, "Prop by method as at latest reg", tables_asat)
    write_tables_to_sheet(writer, "Prop by method delta jun25-latest reg", tables_delta)
    
with pd.ExcelWriter(out_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    table_asat_by_agent.to_excel(writer, sheet_name="Agents as at latest reg", index=False)
    
with pd.ExcelWriter(out_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    table_delta_by_agent.to_excel(writer, sheet_name="Agents delta from jun25 to latest reg", index=False)

with pd.ExcelWriter(out_path, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
    write_tables_to_sheet(writer, "Agents by method as at latest reg",  tables_asat_agent)
    write_tables_to_sheet(writer, "Agents by method delta jun25 to latest reg",  tables_delta_agent)


c:\Users\Pawandeep Kaur\Python\Lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
c:\Users\Pawandeep Kaur\Python\Lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
c:\Users\Pawandeep Kaur\Python\Lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


In [19]:
# Enter the ID and range of the spreadsheet.
spreadsheet_id_avail_iss = '1fqueexmyEQpYi6Q4XzB8zX2VZCu0QOYZdNQjc80da9Y'
sh = gc.open_by_key(spreadsheet_id_avail_iss)

'''worksheets = sh.worksheets()
keep = worksheets[0]  

# delete all other tabs
for ws in worksheets[1:]:
    sh.del_worksheet(ws)

# reset the kept tab
keep.clear()
keep.update_title("sheet1")'''

def get_or_create_ws(sh, title, rows=2000, cols=40):
    existing = {ws.title: ws for ws in sh.worksheets()}
    if title in existing:
        return existing[title]
    return sh.add_worksheet(title=title, rows=str(rows), cols=str(cols))


def df_to_values(df, index_name="Name", include_index=True):
    out = []
    cols = list(df.columns)

    # header row
    out.append(([index_name] if include_index else []) + [str(c) for c in cols])

    # data rows
    for idx, row in df.iterrows():
        out.append(([str(idx)] if include_index else []) + [("" if pd.isna(v) else v) for v in row.tolist()])

    return out

def tables_dict_to_values(tables_dict, index_name="Name", blank_rows_between=2):
    values = []
    for method, tbl in tables_dict.items():
        # method title
        values.append([str(method)])
        values.append([])

        # table
        values.extend(df_to_values(tbl, index_name=index_name))

        # spacing
        for _ in range(blank_rows_between):
            values.append([])

    return values

'''def write_values_to_ws(ws, values):
    ws.clear()
    nrows = max(1, len(values))
    ncols = max(1, max((len(r) for r in values), default=1))
    ws.resize(rows=nrows, cols=ncols)
    ws.update("A1", values, value_input_option="RAW")'''

def write_values_to_ws(ws, values, format_commas=True, header_rows=1):
    ws.clear()
    nrows = max(1, len(values))
    ncols = max(1, max((len(r) for r in values), default=1))
    ws.resize(rows=nrows, cols=ncols)

    # USER_ENTERED helps Sheets treat numbers as numbers
    ws.update("A1", values, value_input_option="USER_ENTERED")

    # apply comma formatting automatically
    if format_commas:
        format_numeric_cells_with_commas(ws, nrows, ncols, header_rows=header_rows)
        
def format_all_numeric_block_except_first_col(ws, nrows, ncols, start_row=1):
    if ncols < 2:
        return
    a1 = f"{rowcol_to_a1(start_row, 2)}:{rowcol_to_a1(nrows, ncols)}"
    ws.format(a1, {"numberFormat": {"type": "NUMBER", "pattern": "#,##0"}})

ws4 = get_or_create_ws(sh, "Prop as at latest reg")
vals4 = df_to_values(table_across_as_at, include_index=False)
write_values_to_ws(ws4, vals4)

ws3 = get_or_create_ws(sh, "Prop delta jun25-latest reg")
vals3 = df_to_values(table_top10_delta, include_index=False)
write_values_to_ws(ws3, vals3)

ws1 = get_or_create_ws(sh, "Prop by method as at latest reg")
vals1 = tables_dict_to_values(tables_asat, index_name="Project Proponent")
write_values_to_ws(ws1, vals1)
format_all_numeric_block_except_first_col(ws1, len(vals1), max(len(r) for r in vals1), start_row=1)

ws2 = get_or_create_ws(sh, "Prop by method delta jun25-latest reg")
vals2 = tables_dict_to_values(tables_delta, index_name="Project Proponent")
write_values_to_ws(ws2, vals2)
format_all_numeric_block_except_first_col(ws2, len(vals2), max(len(r) for r in vals2), start_row=1)



C:\Users\Pawandeep Kaur\AppData\Local\Temp\ipykernel_3144\3924750880.py:66: DeprecationWarning: [Deprecated][in version 6.0.0]: Method signature's arguments 'range_name' and 'values' will change their order. We recommend using named arguments for minimal impact. In addition, the argument 'values' will be mandatory of type: 'List[List]'. (ex) Worksheet.update(values = [[]], range_name=) 
  ws.update("A1", values, value_input_option="USER_ENTERED")


In [22]:
ws = get_or_create_ws(sh, "Agent as at latest reg")
write_values_to_ws(ws, df_to_values(table_asat_by_agent, include_index=False))


ws = get_or_create_ws(sh, "Agent delta jun25-latest reg")
write_values_to_ws(ws, df_to_values(table_delta_by_agent, include_index=False))


ws = get_or_create_ws(sh, "Agent by method as at latest reg")
vals = tables_dict_to_values(tables_asat_agent, index_name="Agent")
write_values_to_ws(ws, vals)
format_all_numeric_block_except_first_col(ws, len(vals), max(len(r) for r in vals), start_row=1)

ws = get_or_create_ws(sh, "Agent by method delta jun25-latest reg")
vals = tables_dict_to_values(tables_delta_agent, index_name="Agent")
write_values_to_ws(ws, vals)
format_all_numeric_block_except_first_col(ws, len(vals), max(len(r) for r in vals), start_row=1)


C:\Users\Pawandeep Kaur\AppData\Local\Temp\ipykernel_3144\3924750880.py:66: DeprecationWarning: [Deprecated][in version 6.0.0]: Method signature's arguments 'range_name' and 'values' will change their order. We recommend using named arguments for minimal impact. In addition, the argument 'values' will be mandatory of type: 'List[List]'. (ex) Worksheet.update(values = [[]], range_name=) 
  ws.update("A1", values, value_input_option="USER_ENTERED")


## Registrations

In [23]:
erf_start = prep_erf(erf_start_raw)
erf_end   = prep_erf(erf_end_raw)

erf_end, display_map   = add_display_and_key(erf_end, COL_PROPONENT, COL_PROPONENT_KEY)
erf_start, _           = add_display_and_key(erf_start, COL_PROPONENT, COL_PROPONENT_KEY)

start_ids = set(erf_start[COL_PROJECT_ID].dropna().astype(str))
end_ids   = set(erf_end[COL_PROJECT_ID].dropna().astype(str))
new_ids   = end_ids - start_ids

new_projects = erf_end[erf_end[COL_PROJECT_ID].astype(str).isin(new_ids)].copy()

totals = (new_projects.groupby(COL_PROPONENT_KEY)[COL_PROJECT_ID]
                    .nunique()
                    .sort_values(ascending=False)
                    .head(15))

top10_list = totals.index.tolist()

method_totals_top10 = (
    new_projects[new_projects[COL_PROPONENT_KEY].isin(top10_list)]
      .groupby(COL_METHOD)[COL_PROJECT_ID]
      .nunique()
      .sort_values(ascending=False)
)

# all method groups present in the dataset
all_method_groups_all = erf_end[COL_METHOD].dropna().unique().tolist()

# sort keys = method groups ranked by top10
sorted_by_top10 = method_totals_top10.index.tolist()
remaining = [m for m in all_method_groups_all if m not in sorted_by_top10]
all_method_groups = sorted_by_top10 + remaining

table_registered_delta = (
    new_projects[new_projects[COL_PROPONENT_KEY].isin(top10_list)]
      .pivot_table(index=COL_PROPONENT_KEY,
                   columns=COL_METHOD,
                   values=COL_PROJECT_ID,
                   aggfunc="nunique",
                   fill_value=0)
      .reindex(index=top10_list, columns=all_method_groups, fill_value=0)
)

table_registered_delta =table_registered_delta.reset_index().rename(columns={"Project Proponent_key": "Project Proponent"})
table_registered_delta


Method Group,Project Proponent,Soil carbon,Plantation forestry,Environmental plantings,Manure management,Savanna fire management (A),Human Induced Regeneration,Savanna fire management (S+A),Energy efficiency,Landfill gas,...,Native Forest from Managed Regrowth,Reforestation and afforestation,Verified Carbon Standard,Fugitives - Coal,Wastewater,Beef cattle herd management,Avoided clearing,Transport - Land and sea,Carbon capture and storage,Blue Carbon
0,AGRIPROVE SOLUTIONS CO NO.2 PTY LTD,45,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,AGRIPROVE SOLUTIONS PTY LTD,31,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,LANDARI PTY LTD,0,9,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,TIMBERLANDS PACIFIC PTY LTD,0,7,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,THE TRUST COMPANY (AUSTRALIA) LIMITED AS TRUST...,0,5,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,FOREST PRODUCTS COMMISSION,0,4,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,ONEFORTYONE PLANTATIONS PTY LTD,0,4,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,LENAH ESTATE PTY LTD,0,3,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,AUSTRALIAN CARBON PRODUCTS PTY LTD,0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,GLENORY NOMINEES PTY LTD AS TRUSTEE FOR C J BU...,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
totals = (erf_end.groupby(COL_PROPONENT_KEY)[COL_PROJECT_ID]
          .nunique()
          .sort_values(ascending=False)
          .head(15))
top10_list = totals.index.tolist()

method_totals_top10 = (
    erf_end[erf_end[COL_PROPONENT_KEY].isin(top10_list)]
      .groupby(COL_METHOD)[COL_PROJECT_ID]
      .nunique()
      .sort_values(ascending=False)
)

all_method_groups_all = erf_end[COL_METHOD].dropna().unique().tolist()
sorted_by_top10 = method_totals_top10.index.tolist()
remaining = [m for m in all_method_groups_all if m not in sorted_by_top10]
all_method_groups = sorted_by_top10 + remaining

table_registered_asat = (
    erf_end[erf_end[COL_PROPONENT_KEY].isin(top10_list)]
      .pivot_table(index=COL_PROPONENT_KEY,
                   columns=COL_METHOD,
                   values=COL_PROJECT_ID,
                   aggfunc="nunique",
                   fill_value=0)
      .reindex(index=top10_list, columns=all_method_groups, fill_value=0)
)

table_registered_asat = table_registered_asat.reset_index().rename(columns={"Project Proponent_key": "Project Proponent"})

table_registered_asat


Method Group,Project Proponent,Soil carbon,Human Induced Regeneration,Landfill gas,Plantation forestry,Environmental plantings,Avoided deforestation,Native Forest from Managed Regrowth,Avoided clearing,Savanna fire management (A),...,Beef cattle herd management,Manure management,Savanna fire management (S+A),Reforestation and afforestation,Verified Carbon Standard,Fugitives - Coal,Wastewater,Transport - Land and sea,Carbon capture and storage,Blue Carbon
0,AGRIPROVE SOLUTIONS PTY LTD,298,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,AGRIPROVE SOLUTIONS CO NO.2 PTY LTD,245,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,TERRA CARBON PTY LIMITED,1,108,0,1,2,31,4,14,9,...,0,1,0,0,0,0,0,0,0,0
3,AGRIPROVE SOLUTIONS CO NO.1 PTY LTD,83,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,LMS ENERGY PTY LTD,0,0,73,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
5,LANDARI PTY LTD,1,0,0,35,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,THE TRUST COMPANY (AUSTRALIA) LIMITED AS TRUST...,0,0,0,32,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,LOAM CARBON PTY LTD,26,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,LGI LIMITED,0,0,24,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,AI CARBON PROJECTS NO 3 PTY LTD,0,23,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
out_path = "registrations.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
#with pd.ExcelWriter(out_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    table_registered_asat.to_excel(writer, sheet_name="Prop as at latest reg", index=True)
    
with pd.ExcelWriter(out_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    table_registered_delta.to_excel(writer, sheet_name="Prop delta latest reg to jun25", index=True)

In [26]:
erf_end, display_map   = add_display_and_key(erf_end, COL_AGENT, GROUP_KEY)
erf_start, _           = add_display_and_key(erf_start, COL_AGENT, GROUP_KEY)
start_ids = set(erf_start[COL_PROJECT_ID].dropna().astype(str))
end_ids   = set(erf_end[COL_PROJECT_ID].dropna().astype(str))
new_ids   = end_ids - start_ids

new_projects = erf_end[erf_end[COL_PROJECT_ID].astype(str).isin(new_ids)].copy()
totals = (new_projects.groupby(GROUP_KEY)[COL_PROJECT_ID]
                    .nunique()
                    .sort_values(ascending=False)
                    .head(15))
totals = totals[~totals.index.isin(["", "UNKNOWN"])]
top10_list = totals.index.tolist()


method_totals_top10 = (
    new_projects[new_projects[GROUP_KEY].isin(top10_list)]
      .groupby(COL_METHOD)[COL_PROJECT_ID]
      .nunique()
      .sort_values(ascending=False)
)

# all method groups present in the dataset
all_method_groups_all = erf_end[COL_METHOD].dropna().unique().tolist()

# sort keys = method groups ranked by top10
sorted_by_top10 = method_totals_top10.index.tolist()
remaining = [m for m in all_method_groups_all if m not in sorted_by_top10]
all_method_groups = sorted_by_top10 + remaining

table_registered_delta_agent = (
    new_projects[new_projects[GROUP_KEY].isin(top10_list)]
      .pivot_table(index=GROUP_KEY,
                   columns=COL_METHOD,
                   values=COL_PROJECT_ID,
                   aggfunc="nunique",
                   fill_value=0)
      .reindex(index=top10_list, columns=all_method_groups, fill_value=0)
)
table_registered_delta_agent =table_registered_delta_agent.reset_index().rename(columns={"Agent_key": "Agent"})
table_registered_delta_agent


Method Group,Agent,Soil carbon,Plantation forestry,Environmental plantings,Manure management,Savanna fire management (A),Human Induced Regeneration,Savanna fire management (S+A),Energy efficiency,Landfill gas,...,Native Forest from Managed Regrowth,Reforestation and afforestation,Verified Carbon Standard,Fugitives - Coal,Wastewater,Beef cattle herd management,Avoided clearing,Transport - Land and sea,Carbon capture and storage,Blue Carbon
0,AGRIPROVE PTY LTD,74,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,SFM ENVIRONMENTAL SOLUTIONS PTY. LTD.,0,12,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CARBON LINK OPERATIONS PTY LTD,11,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,THE TRUSTEE FOR GREEN TRIANGLE FOREST OPERATIN...,0,7,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,PRECISION PASTURES PTY LTD,7,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,MIDWAY PLANTATIONS PTY LTD,0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,SELECT CARBON PTY LTD,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,THE TRUST COMPANY LIMITED AS TRUSTEE FOR ANZLA...,0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,PLANTATION MANAGEMENT PARTNERS PTY LTD,0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,CLIMATE FRIENDLY PTY LTD,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
with pd.ExcelWriter(out_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    table_registered_delta_agent.to_excel(writer, sheet_name="Agent delta jun25 to latest reg", index=True)

In [28]:
erf_end, display_map   = add_display_and_key(erf_end, COL_AGENT, GROUP_KEY)
totals = (erf_end.groupby(GROUP_KEY)[COL_PROJECT_ID]
            .nunique()
            .sort_values(ascending=False)
            .head(15))
totals = totals[~totals.index.isin(["", "UNKNOWN"])]
top10_list = totals.index.tolist()
method_totals_top10 = (
    erf_end[erf_end[GROUP_KEY].isin(top10_list)]
      .groupby(COL_METHOD)[COL_PROJECT_ID]
      .nunique()
      .sort_values(ascending=False)
)

all_method_groups_all = erf_end[COL_METHOD].dropna().unique().tolist()
sorted_by_top10 = method_totals_top10.index.tolist()
remaining = [m for m in all_method_groups_all if m not in sorted_by_top10]
all_method_groups = sorted_by_top10 + remaining

table_registered_asat_agent = (erf_end[erf_end[GROUP_KEY].isin(top10_list)]
    .pivot_table(index=GROUP_KEY, columns="Method Group", values=COL_PROJECT_ID,
                 aggfunc="nunique", fill_value=0)
    .reindex(index=top10_list, columns=all_method_groups, fill_value=0)
)
table_registered_asat_agent =table_registered_asat_agent.reset_index().rename(columns={"Agent_key": "Agent"})
table_registered_asat_agent

Method Group,Agent,Soil carbon,Human Induced Regeneration,Plantation forestry,Environmental plantings,Avoided deforestation,Reforestation and afforestation,Alternative waste treatment,Savanna fire management (A),Energy efficiency,...,Transport - Land and sea,Manure management,Landfill gas,Native Forest from Managed Regrowth,Verified Carbon Standard,Fugitives - Coal,Wastewater,Avoided clearing,Carbon capture and storage,Blue Carbon
0,AGRIPROVE PTY LTD,594,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CLIMATE FRIENDLY PTY LTD,29,134,7,12,4,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,SELECT CARBON PTY LTD,7,54,0,1,3,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CARBON LINK OPERATIONS PTY LTD,55,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,SFM ENVIRONMENTAL SOLUTIONS PTY. LTD.,0,0,38,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,TERRA CARBON PTY LIMITED,0,2,0,0,21,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,CO2 AUSTRALIA PTY LTD,0,5,2,11,0,5,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,THE CARBON FARMING FOUNDATION LTD,3,0,11,9,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,PRECISION PASTURES PTY LTD,20,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,CARBON WEST PTY LTD,17,0,0,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [29]:
with pd.ExcelWriter(out_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    table_registered_asat_agent.to_excel(writer, sheet_name="Agent As at latest reg", index=True)

In [30]:
'''def top10_registrations_by_method_tables(
    df,
    project_id_col,
    proponent_col="Project Proponent",
    method_col="Method Group",
    n=15
):
    mp = (df.groupby([method_col, proponent_col], as_index=False)[project_id_col]
            .nunique()
            .rename(columns={project_id_col: "Registrations"})
            .sort_values([method_col, "Registrations"], ascending=[True, False]))

    tables = {}
    for m, sub in mp.groupby(method_col):
        tbl = (sub.head(n)
               .set_index(proponent_col)[["Registrations"]]
               .rename(columns={"Registrations": str(m)}))  # column header = method name
        tables[m] = tbl

    return tables'''
    
def top10_registrations_by_method_tables(
    df,
    project_id_col,
    proponent_col="Project Proponent",
    method_col="Method Group",
    n=15,
    exclude_unknown=False
):
    d = df.copy()

    # 1) Optionally exclude Unknown / blanks from ranking
    if exclude_unknown:
        key = (d[proponent_col].astype("string")
                 .fillna("")
                 .str.replace(r"\s+", " ", regex=True)
                 .str.strip()
                 .str.upper())
        d = d[~key.isin(["", "UNKNOWN"])].copy()

    # 2) Count unique projects at (method, proponent) level and sort
    mp = (d.groupby([method_col, proponent_col], as_index=False)[project_id_col]
            .nunique()
            .rename(columns={project_id_col: "Registrations"})
            .sort_values([method_col, "Registrations"], ascending=[True, False]))

    # 3) Take top N per method
    tables = {}
    for m, sub in mp.groupby(method_col):
        tbl = (sub.head(n)
               .set_index(proponent_col)[["Registrations"]]
               .rename(columns={"Registrations": str(m)}))  # column header = method name
        tables[m] = tbl

    return tables


tables_reg_asat_prop  = top10_registrations_by_method_tables(erf_end,  project_id_col=COL_PROJECT_ID)
tables_reg_delta_prop = top10_registrations_by_method_tables(new_projects, project_id_col=COL_PROJECT_ID)

tables_reg_asat_agent  = top10_registrations_by_method_tables(erf_end,  project_id_col=COL_PROJECT_ID,proponent_col=COL_AGENT,exclude_unknown=True)
tables_reg_delta_agent = top10_registrations_by_method_tables(new_projects, project_id_col=COL_PROJECT_ID, proponent_col=COL_AGENT,exclude_unknown=True)

In [ ]:
spreadsheet_id_reg = '1x-cmGboq97bdhNbZ1gk6abaGFHgMNWGPvrz7IDELGoA'
sh = gc.open_by_key(spreadsheet_id_reg)

'''worksheets = sh.worksheets()
keep = worksheets[0]  
# delete all other tabs
for ws in worksheets[1:]:
    sh.del_worksheet(ws)
# reset the kept tab
keep.clear()
keep.update_title("sheet1")'''

ws = get_or_create_ws(sh, "Prop As at latest reg")
write_values_to_ws(ws, df_to_values(table_registered_asat, include_index=False))

ws = get_or_create_ws(sh, "Prop Delta jun25-latest reg")
write_values_to_ws(ws, df_to_values(table_registered_delta, include_index=False))

ws2 = get_or_create_ws(sh, "prop by method as at latest reg")
vals2 = tables_dict_to_values(tables_reg_asat_prop, index_name="Project Proponent")
write_values_to_ws(ws2, vals2)

ws2 = get_or_create_ws(sh, "prop by method delta jun25-latest reg")
vals2 = tables_dict_to_values(tables_reg_delta_prop, index_name="Project Proponent")
write_values_to_ws(ws2, vals2)

ws = get_or_create_ws(sh, "Agent As at latest reg")
write_values_to_ws(ws, df_to_values(table_registered_asat_agent, include_index=False))

ws = get_or_create_ws(sh, "Agent Delta jun25-latest reg")
write_values_to_ws(ws, df_to_values(table_registered_delta_agent, include_index=False))

ws2 = get_or_create_ws(sh, "agent by method as at latest reg")
vals2 = tables_dict_to_values(tables_reg_asat_agent, index_name="Agent")
write_values_to_ws(ws2, vals2)

ws2 = get_or_create_ws(sh, "agent by method delta jun25-latest reg")
vals2 = tables_dict_to_values(tables_reg_delta_agent, index_name="Agent")
write_values_to_ws(ws2, vals2)

C:\Users\Pawandeep Kaur\AppData\Local\Temp\ipykernel_3144\3924750880.py:66: DeprecationWarning: [Deprecated][in version 6.0.0]: Method signature's arguments 'range_name' and 'values' will change their order. We recommend using named arguments for minimal impact. In addition, the argument 'values' will be mandatory of type: 'List[List]'. (ex) Worksheet.update(values = [[]], range_name=) 
  ws.update("A1", values, value_input_option="USER_ENTERED")


APIError: {'code': 429, 'message': "Quota exceeded for quota metric 'Write requests' and limit 'Write requests per minute per user' of service 'sheets.googleapis.com' for consumer 'project_number:607272679691'.", 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'RATE_LIMIT_EXCEEDED', 'domain': 'googleapis.com', 'metadata': {'quota_location': 'global', 'service': 'sheets.googleapis.com', 'quota_limit_value': '60', 'consumer': 'projects/607272679691', 'quota_metric': 'sheets.googleapis.com/write_requests', 'quota_unit': '1/min/{project}/{user}', 'quota_limit': 'WriteRequestsPerMinutePerUser'}}, {'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Request a higher quota limit.', 'url': 'https://cloud.google.com/docs/quotas/help/request_increase'}]}]}

### Issuance

In [34]:
start = prep_erf(erf_start_raw)
end   = prep_erf(erf_end_raw)

end_prop, prop_nice = add_display_and_key(end, "Project Proponent", "Project Proponent_key")
start_prop, _            = add_display_and_key(start, "Project Proponent", "Project Proponent_key")

end_agent, agent_nice = add_display_and_key(end, COL_AGENT, GROUP_KEY)
start_agent, _            = add_display_and_key(start, COL_AGENT, GROUP_KEY)

delta = (
    end[[COL_PROJECT_ID, COL_PROPONENT_KEY, COL_PROPONENT, COL_AGENT,GROUP_KEY, COL_METHOD, COL_ISSUED]]
      .merge(start[[COL_PROJECT_ID, COL_ISSUED]], on=COL_PROJECT_ID, how="left", suffixes=("_end", "_start"))
)

delta[f"{COL_ISSUED}_start"] = pd.to_numeric(delta[f"{COL_ISSUED}_start"], errors="coerce").fillna(0)
delta[f"{COL_ISSUED}_end"]   = pd.to_numeric(delta[f"{COL_ISSUED}_end"],   errors="coerce").fillna(0)
delta["Delta issued"] = delta[f"{COL_ISSUED}_end"] - delta[f"{COL_ISSUED}_start"]


In [35]:
'''def top10_across_all_methods(df, key_col, value_col, method_col=COL_METHOD, n=15, key_display_name=None):
    top10 = (df.groupby(key_col, as_index=False)[value_col]
               .sum()
               .sort_values(value_col, ascending=False)
               .head(n)[key_col]
               .tolist())
    method_totals_top10 = (df[df[key_col].isin(top10)]
                           .groupby(method_col)[value_col]
                           .sum())

    all_methods = (
        method_totals_top10
          .reindex(df[method_col].dropna().unique(), fill_value=0)  
          .sort_values(ascending=False)
          .index
          .tolist()
    )

    # pivot
    out = (df[df[key_col].isin(top10)]
           .pivot_table(index=key_col, columns=method_col, values=value_col,
                        aggfunc="sum", fill_value=0)
           .reindex(index=top10, columns=all_methods, fill_value=0)
           .reset_index())
    if key_display_name:
        out = out.rename(columns={key_col: key_display_name})

    out.columns.name = None
    return out'''

def top10_across_all_methods(
    df,
    key_col,
    value_col,
    method_col=COL_METHOD,
    n=15,
    key_display_name=None,
    exclude_unknown=False
):
    d = df.copy()

    if exclude_unknown:
        key_norm = (d[key_col].astype("string")
                      .fillna("")
                      .str.replace(r"\s+", " ", regex=True)
                      .str.strip()
                      .str.upper())
        d = d[~key_norm.isin(["", "UNKNOWN"])].copy()

    top10 = (d.groupby(key_col, as_index=False)[value_col]
               .sum()
               .sort_values(value_col, ascending=False)
               .head(n)[key_col]
               .tolist())

    method_totals_top10 = (d[d[key_col].isin(top10)]
                           .groupby(method_col)[value_col]
                           .sum())

    all_methods = (
        method_totals_top10
          .reindex(d[method_col].dropna().unique(), fill_value=0)
          .sort_values(ascending=False)
          .index
          .tolist()
    )

    out = (d[d[key_col].isin(top10)]
           .pivot_table(index=key_col, columns=method_col, values=value_col,
                        aggfunc="sum", fill_value=0)
           .reindex(index=top10, columns=all_methods, fill_value=0)
           .reset_index())

    if key_display_name:
        out = out.rename(columns={key_col: key_display_name})

    out.columns.name = None
    return out


In [36]:
table_prop_asat  = top10_across_all_methods(
    df=end, key_col=COL_PROPONENT_KEY, value_col=COL_ISSUED, key_display_name="Project Proponent",exclude_unknown=True
)

table_prop_delta = top10_across_all_methods(
    df=delta, key_col=COL_PROPONENT_KEY, value_col="Delta issued", key_display_name="Project Proponent",exclude_unknown=True
)

table_agent_asat  = top10_across_all_methods(
    df=end, key_col=GROUP_KEY, value_col=COL_ISSUED, key_display_name="Agent",exclude_unknown=True
)

table_agent_delta = top10_across_all_methods(
    df=delta, key_col=GROUP_KEY, value_col="Delta issued", key_display_name="Agent",exclude_unknown=True
)


In [37]:
'''def top10_by_method_tables(df, value_col,
                           key_col,                
                           display_col,             
                           method_col="Method Group",
                           n=15):

    mp = (df.groupby([method_col, key_col, display_col], as_index=False)[value_col]
            .sum()
            .sort_values([method_col, value_col], ascending=[True, False]))

    tables = {}
    for m, sub in mp.groupby(method_col):
        tbl = (sub.head(n)
               .set_index(display_col)[[value_col]]
               .rename(columns={value_col: str(m)}))   
        tables[m] = tbl

    return tables
'''

def top10_by_method_tables(
    df,
    value_col,
    key_col,
    display_col,
    method_col="Method Group",
    n=15,
    exclude_unknown=False
):
    d = df.copy()
    
    if exclude_unknown:
        key_norm = (d[key_col].astype("string")
                      .fillna("")
                      .str.replace(r"\s+", " ", regex=True)
                      .str.strip()
                      .str.upper())
        d = d[~key_norm.isin(["", "UNKNOWN"])].copy()

    mp = (d.groupby([method_col, key_col, display_col], as_index=False)[value_col]
            .sum()
            .sort_values([method_col, value_col], ascending=[True, False]))

    tables = {}
    for m, sub in mp.groupby(method_col):
        tbl = (sub.head(n)
               .set_index(display_col)[[value_col]]
               .rename(columns={value_col: str(m)}))
        tables[m] = tbl

    return tables

tables_prop_asat = top10_by_method_tables(
    df=end_prop, value_col=COL_ISSUED,
    key_col="Project Proponent_key", display_col="Project Proponent",exclude_unknown=True
)

tables_prop_delta = top10_by_method_tables(
    df=delta, value_col="Delta issued",
    key_col="Project Proponent_key", display_col="Project Proponent",exclude_unknown=True
)

tables_agent_asat = top10_by_method_tables(
    df=end_agent, value_col=COL_ISSUED,
    key_col=GROUP_KEY, display_col=COL_AGENT,exclude_unknown=True
)

tables_agent_delta = top10_by_method_tables(
    df=delta, value_col="Delta issued",
    key_col=GROUP_KEY, display_col=COL_AGENT,exclude_unknown=True
)



In [ ]:
# Enter the ID and range of the spreadsheet.
spreadsheet_id_iss = '1Fyfhp4eHz9RSZqrQeSAT4PwIyidshG-qHVRUZuF2S4o'
sh = gc.open_by_key(spreadsheet_id_iss)

ws4 = get_or_create_ws(sh, "Prop as at latest reg")
vals4 = df_to_values(table_prop_asat, include_index=False)
write_values_to_ws(ws4, vals4)

ws3 = get_or_create_ws(sh, "Prop delta jun25-latest reg")
vals3 = df_to_values(table_prop_delta, include_index=False)
write_values_to_ws(ws3, vals3)

ws = get_or_create_ws(sh, "Agent as at latest reg")
write_values_to_ws(ws, df_to_values(table_agent_asat, include_index=False))

ws = get_or_create_ws(sh, "Agent delta jun25-latest reg")
write_values_to_ws(ws, df_to_values(table_agent_delta, include_index=False))

C:\Users\Pawandeep Kaur\AppData\Local\Temp\ipykernel_3144\3924750880.py:66: DeprecationWarning: [Deprecated][in version 6.0.0]: Method signature's arguments 'range_name' and 'values' will change their order. We recommend using named arguments for minimal impact. In addition, the argument 'values' will be mandatory of type: 'List[List]'. (ex) Worksheet.update(values = [[]], range_name=) 
  ws.update("A1", values, value_input_option="USER_ENTERED")


APIError: {'code': 429, 'message': "Quota exceeded for quota metric 'Write requests' and limit 'Write requests per minute per user' of service 'sheets.googleapis.com' for consumer 'project_number:765823740141'.", 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'RATE_LIMIT_EXCEEDED', 'domain': 'googleapis.com', 'metadata': {'consumer': 'projects/765823740141', 'quota_limit_value': '60', 'service': 'sheets.googleapis.com', 'quota_unit': '1/min/{project}/{user}', 'quota_metric': 'sheets.googleapis.com/write_requests', 'quota_location': 'global', 'quota_limit': 'WriteRequestsPerMinutePerUser'}}, {'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Request a higher quota limit.', 'url': 'https://cloud.google.com/docs/quotas/help/request_increase'}]}]}

In [56]:
ws1 = get_or_create_ws(sh, "Prop by method as at latest reg")
vals1 = tables_dict_to_values(tables_prop_asat, index_name="Project Proponent")
write_values_to_ws(ws1, vals1)
format_all_numeric_block_except_first_col(ws1, len(vals1), max(len(r) for r in vals1), start_row=1)

ws2 = get_or_create_ws(sh, "Prop by method delta jun25-latest reg")
vals2 = tables_dict_to_values(tables_prop_delta, index_name="Project Proponent")
write_values_to_ws(ws2, vals2)
format_all_numeric_block_except_first_col(ws2, len(vals2), max(len(r) for r in vals2), start_row=1
                                          )
ws = get_or_create_ws(sh, "Agent by method as at latest reg")
vals = tables_dict_to_values(tables_agent_asat, index_name="Agent")
write_values_to_ws(ws, vals)
format_all_numeric_block_except_first_col(ws, len(vals), max(len(r) for r in vals), start_row=1)

ws = get_or_create_ws(sh, "Agent by method delta jun25-latest reg")
vals = tables_dict_to_values(tables_agent_delta, index_name="Agent")
write_values_to_ws(ws, vals)

format_all_numeric_block_except_first_col(ws, len(vals), max(len(r) for r in vals), start_row=1)

C:\Users\Pawandeep Kaur\AppData\Local\Temp\ipykernel_3144\3924750880.py:66: DeprecationWarning: [Deprecated][in version 6.0.0]: Method signature's arguments 'range_name' and 'values' will change their order. We recommend using named arguments for minimal impact. In addition, the argument 'values' will be mandatory of type: 'List[List]'. (ex) Worksheet.update(values = [[]], range_name=) 
  ws.update("A1", values, value_input_option="USER_ENTERED")


## CAC analysis

In [11]:
cac_12_2024_fp = (
    Path.home()/"RepuTex"/"Company - Documents"/"3 Research"/"2 Models and Databases"/"9 Registry Database"/"CAC Registry" / "CAC_Register_2024-12-31.xlsx"
)

#cac_12_2024 = pd.read_excel('C:\\Users\\Pawandeep Kaur\\RepuTex\\Company - Documents\\3 Research\\2 Models and Databases\\9 Registry Database\\CAC Registry\\CAC_Register_2024-12-31.xlsx', header=2)
cac_12_2024 = pd.read_excel(cac_12_2024_fp, header=2)
#cac_12_2024 = cac_12_2024.drop(columns=["Column1"])
cac_12_2024.head(2)


,Carbon Abatement Contract ID,Contractor Name,Project(s),Project ID,Delivery Obligation,Original volume of abatement committed under contract,Volume released through Fixed Delivery exit arrangements or lapsed Optional Delivery milestones,Volume of abatement sold to the Commonwealth under contract,Delivery period in years (as set out in the contract),"Total contract length, including time to fulfil Conditions Precedent (as set out in the contract)",Actual contract duration from commencement date,Actual contract duration from contract date,Subject to Conditions Precedent at time of contract,Actual contract end date,Status,Auction date
0,CAC100911,Terra Carbon Pty Limited,Maranoa Ecosystem Conservation Project,ERF101930,Fixed Delivery,179818,8105,146129,10,10,-,-,No,-,Active,2016-04-01
1,CAC101886,LMS Energy Pty Ltd,Rockhampton Landfill Gas Project,ERF157317,Fixed Delivery,149000,21000,42398,7,7,-,-,No,-,Active,2021-04-01


In [12]:
cac_june_2025_fp = (
    Path.home()/"RepuTex"/"Company - Documents"/"3 Research"/"2 Models and Databases"/"9 Registry Database"/"CAC Registry" / "CAC_Register_2025-06-30.xlsx"
)
cac_jun_2025 = pd.read_excel(cac_june_2025_fp, header=2)
#cac_jun_2025 = pd.read_excel('C:\\Users\\Pawandeep Kaur\\RepuTex\\Company - Documents\\3 Research\\2 Models and Databases\\9 Registry Database\\CAC Registry\\CAC_Register_2025-06-30.xlsx', header=2)
cac_jun_2025.head(2)

c:\Users\Pawandeep Kaur\Python\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,Carbon Abatement Contract ID,Contractor Name,Project(s),Project ID,Delivery Obligation,Original volume of abatement committed under contract,Volume released through Fixed Delivery exit arrangements or lapsed Optional Delivery milestones,Volume of abatement sold to the Commonwealth under contract,Delivery period in years (as set out in the contract),"Total contract length, including time to fulfil Conditions Precedent (as set out in the contract)",Actual contract duration from commencement date,Actual contract duration from contract date,Subject to Conditions Precedent at time of contract,Actual contract end date,Status,Auction date
0,CAC100911,Terra Carbon Pty Limited,Maranoa Ecosystem Conservation Project,ERF101930,Fixed Delivery,179818,8105,146129,16,16,-,-,No,-,Active,2016-04-01
1,CAC101886,LMS Energy Pty Ltd,Rockhampton Landfill Gas Project,ERF157317,Fixed Delivery,149000,21000,42398,7,7,-,-,No,-,Active,2021-04-01


In [59]:
cac_latest_url = 'https://cer.gov.au/document/carbon-abatement-contract-register'

# Read the Excel file, handling missing values
cac_latest = pd.read_excel(cac_latest_url, header=2)

cac_latest.head(2)

c:\Users\Pawandeep Kaur\Python\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,Carbon Abatement Contract ID,Contractor Name,Project(s),Project ID,Delivery Obligation,Original volume of abatement committed under contract,Volume released through Fixed Delivery exit arrangements or lapsed Optional Delivery milestones,Volume of abatement sold to the Commonwealth under contract,Delivery period in years (as set out in the contract),"Total contract length, including time to fulfil Conditions Precedent (as set out in the contract)",Actual contract duration from commencement date,Actual contract duration from contract date,Subject to Conditions Precedent at time of contract,Actual contract end date,Status,Auction date
0,CAC100911,Terra Carbon Pty Limited,Maranoa Ecosystem Conservation Project,ERF101930,Fixed Delivery,179818,8105,146129,16,16,-,-,No,-,Active,2016-04-01
1,CAC101886,LMS Energy Pty Ltd,Rockhampton Landfill Gas Project,ERF157317,Fixed Delivery,149000,21000,63799,7,7,-,-,No,-,Active,2021-04-01


In [60]:
col = "Volume of abatement sold to the Commonwealth under contract"
ID_COL = "Carbon Abatement Contract ID" 


cac_12_2024 = cac_12_2024.merge(
    cac_latest[[ID_COL, col]].rename(columns={col: f"{col} (latest register)"}),
    on=ID_COL, how="left"
)

cac_12_2024 = cac_12_2024.merge(
    cac_jun_2025[[ID_COL, col]].rename(columns={col: f"{col} (june register)"}),
    on=ID_COL, how="left"
)

# bring the latest column into the Dec-2024 dataframe (keep original Dec values too)
#cac_12_2024[f"{col} (latest register)"] = cac_latest[col].values
#cac_12_2024[f"{col} (june register)"] = cac_jun_2025[col].values
# difference = latest - dec2024
cac_12_2024["CAC deliveries from 1 jan 2025"] = (
    pd.to_numeric(cac_12_2024[f"{col} (latest register)"], errors="coerce").fillna(0)
    - pd.to_numeric(cac_12_2024[col], errors="coerce").fillna(0)
)

cac_12_2024["CAC deliveries from 1 jan 2025 as at june25"] = (
    pd.to_numeric(cac_12_2024[f"{col} (june register)"], errors="coerce").fillna(0)
    - pd.to_numeric(cac_12_2024[col], errors="coerce").fillna(0)
)
cac_transformed = cac_12_2024.copy()

cac_transformed.head(2)

,Carbon Abatement Contract ID,Contractor Name,Project(s),Project ID,Delivery Obligation,Original volume of abatement committed under contract,Volume released through Fixed Delivery exit arrangements or lapsed Optional Delivery milestones,Volume of abatement sold to the Commonwealth under contract,Delivery period in years (as set out in the contract),"Total contract length, including time to fulfil Conditions Precedent (as set out in the contract)",Actual contract duration from commencement date,Actual contract duration from contract date,Subject to Conditions Precedent at time of contract,Actual contract end date,Status,Auction date,Volume of abatement sold to the Commonwealth under contract (latest register),Volume of abatement sold to the Commonwealth under contract (june register),CAC deliveries from 1 jan 2025,CAC deliveries from 1 jan 2025 as at june25
0,CAC100911,Terra Carbon Pty Limited,Maranoa Ecosystem Conservation Project,ERF101930,Fixed Delivery,179818,8105,146129,10,10,-,-,No,-,Active,2016-04-01,146129,146129,0,0
1,CAC101886,LMS Energy Pty Ltd,Rockhampton Landfill Gas Project,ERF157317,Fixed Delivery,149000,21000,42398,7,7,-,-,No,-,Active,2021-04-01,63799,42398,21401,0


In [61]:
COL_CONTRACTED = "Original volume of abatement committed under contract"
COL_DELIVERED  = "Volume of abatement sold to the Commonwealth under contract"
COL_RELEASED   = "Volume released through Fixed Delivery exit arrangements or lapsed Optional Delivery milestones"
COL_STATUS     = "Status"

def add_outstanding(df):
    # make sure numeric
    for c in [COL_CONTRACTED, COL_DELIVERED, COL_RELEASED]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    # lapsed at CAC-id level (row-level): only for Lapsed/Terminated rows
    df["Lapsed volume"] = np.where(df[COL_STATUS].eq("Lapsed/Terminated"), df[COL_CONTRACTED], 0)

    # outstanding
    df["Outstanding contract volume as at 31-12-2024"] = (
        df[COL_CONTRACTED] - df[COL_DELIVERED] - df[COL_RELEASED] - df["Lapsed volume"]
    )
    return df

# outstanding as at 31 Dec 2024 
cac_transformed = add_outstanding(cac_transformed)

cac_transformed.head(2)


,Carbon Abatement Contract ID,Contractor Name,Project(s),Project ID,Delivery Obligation,Original volume of abatement committed under contract,Volume released through Fixed Delivery exit arrangements or lapsed Optional Delivery milestones,Volume of abatement sold to the Commonwealth under contract,Delivery period in years (as set out in the contract),"Total contract length, including time to fulfil Conditions Precedent (as set out in the contract)",...,Subject to Conditions Precedent at time of contract,Actual contract end date,Status,Auction date,Volume of abatement sold to the Commonwealth under contract (latest register),Volume of abatement sold to the Commonwealth under contract (june register),CAC deliveries from 1 jan 2025,CAC deliveries from 1 jan 2025 as at june25,Lapsed volume,Outstanding contract volume as at 31-12-2024
0,CAC100911,Terra Carbon Pty Limited,Maranoa Ecosystem Conservation Project,ERF101930,Fixed Delivery,179818,8105,146129,10,10,...,No,-,Active,2016-04-01,146129,146129,0,0,0,25584
1,CAC101886,LMS Energy Pty Ltd,Rockhampton Landfill Gas Project,ERF157317,Fixed Delivery,149000,21000,42398,7,7,...,No,-,Active,2021-04-01,63799,42398,21401,0,0,85602


In [62]:
outstanding_del = "Outstanding contract volume as at 31-12-2024"
latest_delivered = "CAC deliveries from 1 jan 2025"
june_delivered = "CAC deliveries from 1 jan 2025 as at june25" 

mask = (
    cac_transformed["Delivery Obligation"].astype(str).str.strip().eq("Fixed Delivery")
    & cac_transformed["Status"].astype(str).str.strip().eq("Active")
)

base = (
    0.25 * pd.to_numeric(cac_transformed[outstanding_del], errors="coerce").fillna(0)
    - pd.to_numeric(cac_transformed[latest_delivered], errors="coerce").fillna(0)
)

base2 = (
    0.25 * pd.to_numeric(cac_transformed[outstanding_del], errors="coerce").fillna(0)
    - pd.to_numeric(cac_transformed[june_delivered], errors="coerce").fillna(0)
)


cac_transformed["Min Delivery Criteria as at latest register"] = np.where(mask, base, 0)

cac_transformed["Min Delivery Criteria as at june register"] = np.where(mask, base2, 0)

cac_transformed["Min Delivery Delta"] = (
    cac_transformed["Min Delivery Criteria as at latest register"].fillna(0)
    - cac_transformed["Min Delivery Criteria as at june register"].fillna(0)
)


cac_transformed.to_excel("cac_analysis.xlsx", index=False)

In [63]:

erf_url = 'https://cer.gov.au/document/accu-scheme-project-registerxlsx'
erf_latest = pd.read_excel(erf_url, header = 3)


erf_latest = add_method_group(erf_latest, method_keywords, method_mapping, method_col="Method")

erf_latest = erf_latest[
    erf_latest["Project revoked"].astype("string")
        .fillna("")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.casefold()
        .eq("no")
].copy()

erf_latest.head(2)


,Project Proponent,Agent,"Significantly Involved Person(s), Description of involvement",Project Name,Project ID,Method,Method URL,Method Type,Estimation or measurement approach or model,Project Description,...,NKACCUs Units Issued in Financial Year 2023/24,NKACCUs Units Issued in Financial Year 2024/25,NKACCUs Units Issued in Financial Year 2025/26,Name of person/s to whom the NKACCUs issued,Total Number of NKACCUs units relinquished,Enforceable undertaking URLs related to this project,Enforceable undertakings related to this project,Notes,Project revoked,Method Group
0,AGRIPROVE SOLUTIONS CO NO.2 PTY LTD,Agriprove Pty Ltd,NaN,Yona SOC project,ERF191531,Carbon Credits (Carbon Farming Initiative-Esti...,https://www.legislation.gov.au/Details/F2021L0...,Agriculture,NaN,This project increases carbon in soil in the a...,...,0,0,0,NaN,0,NaN,NaN,NaN,No,Soil carbon
1,Glenory Nominees Pty Ltd as Trustee for C J Bu...,Select Carbon Pty Ltd,NaN,New Horizon Soil Carbon Project,ERF204747,Carbon Credits (Carbon Farming Initiative-Esti...,https://www.legislation.gov.au/Details/F2021L0...,Agriculture,NaN,This project increases carbon in soil in the a...,...,0,0,0,NaN,0,NaN,NaN,NaN,No,Soil carbon


#### erf and cac merged on project id

In [64]:
KEY = "Project ID"

def split_project_ids(x):
    if pd.isna(x):
        return []
    s = str(x).replace("\r", "\n")
    # split on newlines; (add extra splits if you ever see commas/semicolons)
    parts = [p.strip() for p in s.split("\n")]
    return [p for p in parts if p]

cac_expanded = (
    cac_transformed
      .assign(**{KEY: cac_transformed[KEY].apply(split_project_ids)})
      .explode(KEY)
)


cac_expanded[KEY] = cac_expanded[KEY].astype(str).str.strip()

cac_expanded.head(2)

,Carbon Abatement Contract ID,Contractor Name,Project(s),Project ID,Delivery Obligation,Original volume of abatement committed under contract,Volume released through Fixed Delivery exit arrangements or lapsed Optional Delivery milestones,Volume of abatement sold to the Commonwealth under contract,Delivery period in years (as set out in the contract),"Total contract length, including time to fulfil Conditions Precedent (as set out in the contract)",...,Auction date,Volume of abatement sold to the Commonwealth under contract (latest register),Volume of abatement sold to the Commonwealth under contract (june register),CAC deliveries from 1 jan 2025,CAC deliveries from 1 jan 2025 as at june25,Lapsed volume,Outstanding contract volume as at 31-12-2024,Min Delivery Criteria as at latest register,Min Delivery Criteria as at june register,Min Delivery Delta
0,CAC100911,Terra Carbon Pty Limited,Maranoa Ecosystem Conservation Project,ERF101930,Fixed Delivery,179818,8105,146129,10,10,...,2016-04-01,146129,146129,0,0,0,25584,"6,396","6,396",0
1,CAC101886,LMS Energy Pty Ltd,Rockhampton Landfill Gas Project,ERF157317,Fixed Delivery,149000,21000,42398,7,7,...,2021-04-01,63799,42398,21401,0,0,85602,-0,"21,400","-21,401"


In [65]:
merged_project_level = erf_latest.merge(
    cac_expanded,
    on=KEY,
    how="left",
    suffixes=("", "_cac")   
)

merged_project_level.head(2)

,Project Proponent,Agent,"Significantly Involved Person(s), Description of involvement",Project Name,Project ID,Method,Method URL,Method Type,Estimation or measurement approach or model,Project Description,...,Auction date,Volume of abatement sold to the Commonwealth under contract (latest register),Volume of abatement sold to the Commonwealth under contract (june register),CAC deliveries from 1 jan 2025,CAC deliveries from 1 jan 2025 as at june25,Lapsed volume,Outstanding contract volume as at 31-12-2024,Min Delivery Criteria as at latest register,Min Delivery Criteria as at june register,Min Delivery Delta
0,AGRIPROVE SOLUTIONS CO NO.2 PTY LTD,Agriprove Pty Ltd,NaN,Yona SOC project,ERF191531,Carbon Credits (Carbon Farming Initiative-Esti...,https://www.legislation.gov.au/Details/F2021L0...,Agriculture,NaN,This project increases carbon in soil in the a...,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Glenory Nominees Pty Ltd as Trustee for C J Bu...,Select Carbon Pty Ltd,NaN,New Horizon Soil Carbon Project,ERF204747,Carbon Credits (Carbon Farming Initiative-Esti...,https://www.legislation.gov.au/Details/F2021L0...,Agriculture,NaN,This project increases carbon in soil in the a...,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
with pd.ExcelWriter('cac_analysis.xlsx', engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    merged_project_level.to_excel(writer, sheet_name="merged_project_level", index=False)

In [ ]:
COL_VAL    = "Min Delivery Criteria as at latest register"
df = merged_project_level.copy()
df["Project Proponent_key"] = (
    df[COL_PROPONENT]
      .astype(str)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
      .str.casefold()
)
display_map = (
    df.loc[df[COL_PROPONENT].notna(), ["Project Proponent_key", COL_PROPONENT]]
      .assign(**{COL_PROPONENT: lambda x: x[COL_PROPONENT].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()})
      .groupby("Project Proponent_key")[COL_PROPONENT]
      .agg(lambda s: s.value_counts().index[0])
)

df[COL_VAL] = pd.to_numeric(df[COL_VAL], errors="coerce").fillna(0)

pivot_prop_method = (
    pd.pivot_table(
        df,
        index="Project Proponent_key",
        columns=COL_METHOD,
        values=COL_VAL,
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)


pivot_prop_method[COL_PROPONENT] = pivot_prop_method["Project Proponent_key"].map(display_map)

# put display name first (optional)
cols = [COL_PROPONENT, "Project Proponent_key"] + [c for c in pivot_prop_method.columns if c not in [COL_PROPONENT, "Project Proponent_key"]]
pivot_prop_method = pivot_prop_method[cols].drop(columns="Project Proponent_key")

method_cols = [c for c in pivot_prop_method.columns if c != COL_PROPONENT]
pivot_prop_method["__total__"] = pivot_prop_method[method_cols].sum(axis=1)

top10 = (
    pivot_prop_method.sort_values("__total__", ascending=False)
    .head(15)
    .drop(columns="__total__")
)

top10.columns.name = None 

method_cols = [c for c in top10.columns if c != COL_PROPONENT]

col_order = (
    top10[method_cols]
      .sum(axis=0)
      .sort_values(ascending=False)
      .index
      .tolist()
)

top10 = top10[[COL_PROPONENT] + col_order]

top10.head(2)

,Project Proponent,Soil carbon,Human Induced Regeneration,Avoided deforestation,Savanna fire management (A),Energy efficiency,Alternative waste treatment,Environmental plantings,Transport - Land and sea,Avoided clearing,...,Fugitives - Coal,Carbon capture and storage,Native Forest from Managed Regrowth,Manure management,Landfill gas,Plantation forestry,Savanna fire management (S+A),Reforestation and afforestation,Verified Carbon Standard,Wastewater
16,AGRIPROVE SOLUTIONS PTY LTD,"13,907,202",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
266,Corporate Carbon Solutions Pty Ltd,"1,879,250","317,655",0,0,"139,120","68,000","66,500","58,329",0,...,0,0,0,0,0,0,0,0,0,0


In [68]:
mp = (df.groupby([COL_METHOD, "Project Proponent_key"], as_index=False)[COL_VAL]
        .sum()
        .sort_values([COL_METHOD, COL_VAL], ascending=[True, False]))

tables_top10_by_method = {}
for m, sub in mp.groupby(COL_METHOD):
    tbl = (sub.head(15)
             .assign(**{COL_PROPONENT: sub["Project Proponent_key"].map(display_map)})
             .set_index(COL_PROPONENT)[[COL_VAL]]
             .rename(columns={COL_VAL: str(m)}))     # column name = method
    tables_top10_by_method[m] = tbl



In [69]:
COL_VAL_DELTA    = "Min Delivery Delta"

df = merged_project_level.copy()

df["Project Proponent_key"] = (
    df[COL_PROPONENT]
      .astype(str)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
      .str.casefold()
)

display_map = (
    df.loc[df[COL_PROPONENT].notna(), ["Project Proponent_key", COL_PROPONENT]]
      .assign(**{COL_PROPONENT: lambda x: x[COL_PROPONENT].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()})
      .groupby("Project Proponent_key")[COL_PROPONENT]
      .agg(lambda s: s.value_counts().index[0])
)

df[COL_VAL_DELTA] = pd.to_numeric(df[COL_VAL_DELTA], errors="coerce").fillna(0)

pivot_prop_method = (
    pd.pivot_table(
        df,
        index="Project Proponent_key",
        columns=COL_METHOD,
        values=COL_VAL_DELTA,
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)


pivot_prop_method[COL_PROPONENT] = pivot_prop_method["Project Proponent_key"].map(display_map)

cols = [COL_PROPONENT, "Project Proponent_key"] + [c for c in pivot_prop_method.columns if c not in [COL_PROPONENT, "Project Proponent_key"]]
pivot_prop_method = pivot_prop_method[cols].drop(columns="Project Proponent_key")

method_cols = [c for c in pivot_prop_method.columns if c != COL_PROPONENT]
pivot_prop_method["__total__"] = pivot_prop_method[method_cols].sum(axis=1)

top10_delta = (
    pivot_prop_method.sort_values("__total__", ascending=False)
    .head(15)
    .drop(columns="__total__")
)

top10_delta.columns.name = None 

method_cols = [c for c in top10_delta.columns if c != COL_PROPONENT]

col_order = (
    top10_delta[method_cols]
      .sum(axis=0)
      .sort_values(ascending=False)
      .index
      .tolist()
)

top10_delta = top10_delta[[COL_PROPONENT] + col_order]

top10_delta.head(2)

,Project Proponent,Alternative waste treatment,Avoided clearing,Avoided deforestation,Beef cattle herd management,Blue Carbon,Carbon capture and storage,Energy efficiency,Environmental plantings,Fugitives - Coal,...,Manure management,Native Forest from Managed Regrowth,Plantation forestry,Reforestation and afforestation,Savanna fire management (A),Savanna fire management (S+A),Soil carbon,Transport - Land and sea,Verified Carbon Standard,Wastewater
984,ZENITH AUSTRALIA INVESTMENT HOLDING PTY LTD,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0,24 DEGREE FOREST PTY LTD,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [70]:
mp_delta = (df.groupby([COL_METHOD, "Project Proponent_key"], as_index=False)[COL_VAL]
        .sum()
        .sort_values([COL_METHOD, COL_VAL], ascending=[True, False]))

tables_top10_by_method_delta = {}
for m, sub in mp_delta.groupby(COL_METHOD):
    tbl = (sub.head(15)
             .assign(**{COL_PROPONENT: sub["Project Proponent_key"].map(display_map)})
             .set_index(COL_PROPONENT)[[COL_VAL]]
             .rename(columns={COL_VAL: str(m)}))     # column name = method
    tables_top10_by_method_delta[m] = tbl


In [71]:
spreadsheet_id_min = '1VGXIgGqc_Q0nFOiEIFe3x9tEQbTB0PngjUZSsbwUE6c'
sh = gc.open_by_key(spreadsheet_id_min)
vals = [top10.columns.tolist()] + top10.astype(object).where(top10.notna(), "").values.tolist()
vals2 = [top10_delta.columns.tolist()] + top10_delta.astype(object).where(top10_delta.notna(), "").values.tolist()
ws = get_or_create_ws(sh, "Prop as at latest reg")
write_values_to_ws(ws, vals)

ws2 = get_or_create_ws(sh, "Prop delta jun25-latest reg")
write_values_to_ws(ws2, vals2)

ws3 = get_or_create_ws(sh, "Prop by method as at latest reg")
vals3 = tables_dict_to_values(tables_top10_by_method, index_name="Project Proponent")
write_values_to_ws(ws3, vals3)
format_all_numeric_block_except_first_col(ws3, len(vals3), max(len(r) for r in vals3), start_row=1)

ws4 = get_or_create_ws(sh, "Prop by method delta jun25-latest reg")
vals4 = tables_dict_to_values(tables_top10_by_method_delta, index_name="Project Proponent")
write_values_to_ws(ws4, vals4)
format_all_numeric_block_except_first_col(ws4, len(vals4), max(len(r) for r in vals4), start_row=1)

C:\Users\Pawandeep Kaur\AppData\Local\Temp\ipykernel_3144\3924750880.py:66: DeprecationWarning: [Deprecated][in version 6.0.0]: Method signature's arguments 'range_name' and 'values' will change their order. We recommend using named arguments for minimal impact. In addition, the argument 'values' will be mandatory of type: 'List[List]'. (ex) Worksheet.update(values = [[]], range_name=) 
  ws.update("A1", values, value_input_option="USER_ENTERED")


In [72]:
df = merged_project_level.copy()
df["Agent_key"] = (
    df[COL_AGENT]
      .astype(str)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
      .str.casefold()
)


display_map = (
    df.loc[df[COL_AGENT].notna(), ["Agent_key", COL_AGENT]]
      .assign(**{COL_AGENT: lambda x: x[COL_AGENT].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()})
      .groupby("Agent_key")[COL_AGENT]
      .agg(lambda s: s.value_counts().index[0])
)

df[COL_VAL] = pd.to_numeric(df[COL_VAL], errors="coerce").fillna(0)

pivot_prop_method = (
    pd.pivot_table(
        df,
        index="Agent_key",
        columns=COL_METHOD,
        values=COL_VAL,
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

pivot_prop_method[COL_AGENT] = pivot_prop_method["Agent_key"].map(display_map)
cols = [COL_AGENT, "Agent_key"] + [c for c in pivot_prop_method.columns if c not in [COL_AGENT, "Agent_key"]]
pivot_prop_method = pivot_prop_method[cols].drop(columns="Agent_key")

method_cols = [c for c in pivot_prop_method.columns if c != COL_AGENT]
pivot_prop_method["__total__"] = pivot_prop_method[method_cols].sum(axis=1)

# --- drop Unknown/blank agents before taking Top N ---
pivot_prop_method[COL_AGENT] = pivot_prop_method[COL_AGENT].astype("string").fillna("")

mask_unknown = (
    pivot_prop_method[COL_AGENT].str.strip().str.casefold().eq("unknown") |
    pivot_prop_method[COL_AGENT].str.strip().eq("")
)

pivot_prop_method = pivot_prop_method[~mask_unknown].copy()


top10 = (
    pivot_prop_method.sort_values("__total__", ascending=False)
    .head(15)
    .drop(columns="__total__")
)

top10.columns.name = None 
col = "Agent"
top10[col] = top10[col].replace(r"^\s*$", np.nan, regex=True).fillna("Unknown")

method_cols = [c for c in top10.columns if c != COL_AGENT]

col_order = (
    top10[method_cols]
      .sum(axis=0)
      .sort_values(ascending=False)
      .index
      .tolist()
)

top10 = top10[[COL_AGENT] + col_order]
top10.head(2)


,Agent,Soil carbon,Human Induced Regeneration,Avoided deforestation,Energy efficiency,Environmental plantings,Alternative waste treatment,Transport - Land and sea,Savanna fire management (A),Beef cattle herd management,...,Blue Carbon,Carbon capture and storage,Native Forest from Managed Regrowth,Manure management,Landfill gas,Fugitives - Coal,Savanna fire management (S+A),Reforestation and afforestation,Verified Carbon Standard,Wastewater
2,Agriprove Pty Ltd,"15,786,452",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
29,Climate Friendly Pty Ltd,"5,000","4,165,457","20,932",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [73]:
df = merged_project_level.copy()
df["Agent_key"] = (
    df[COL_AGENT]
      .astype(str)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
      .str.casefold()
)

display_map = (
    df.loc[df[COL_AGENT].notna(), ["Agent_key", COL_AGENT]]
      .assign(**{COL_AGENT: lambda x: x[COL_AGENT].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()})
      .groupby("Agent_key")[COL_AGENT]
      .agg(lambda s: s.value_counts().index[0])
)

df[COL_VAL_DELTA] = pd.to_numeric(df[COL_VAL_DELTA], errors="coerce").fillna(0)

pivot_prop_method = (
    pd.pivot_table(
        df,
        index="Agent_key",
        columns=COL_METHOD,
        values=COL_VAL_DELTA,
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

pivot_prop_method[COL_AGENT] = pivot_prop_method["Agent_key"].map(display_map)
cols = [COL_AGENT, "Agent_key"] + [c for c in pivot_prop_method.columns if c not in [COL_AGENT, "Agent_key"]]
pivot_prop_method = pivot_prop_method[cols].drop(columns="Agent_key")

method_cols = [c for c in pivot_prop_method.columns if c != COL_AGENT]
pivot_prop_method["__total__"] = pivot_prop_method[method_cols].sum(axis=1)

# --- drop Unknown/blank agents before taking Top N ---
pivot_prop_method[COL_AGENT] = pivot_prop_method[COL_AGENT].astype("string").fillna("")

mask_unknown = (
    pivot_prop_method[COL_AGENT].str.strip().str.casefold().eq("unknown") |
    pivot_prop_method[COL_AGENT].str.strip().eq("")
)

pivot_prop_method = pivot_prop_method[~mask_unknown].copy()


top10_delta_agent = (
    pivot_prop_method.sort_values("__total__", ascending=False)
    .head(15)
    .drop(columns="__total__")
)

top10_delta_agent.columns.name = None 
col = "Agent"
top10_delta_agent[col] = top10_delta_agent[col].replace(r"^\s*$", np.nan, regex=True).fillna("Unknown")

method_cols = [c for c in top10.columns if c != COL_AGENT]

col_order = (
    top10_delta_agent[method_cols]
      .sum(axis=0)
      .sort_values(ascending=False)
      .index
      .tolist()
)

top10_delta_agent = top10_delta_agent[[COL_AGENT] + col_order]
top10_delta_agent.head(2)


,Agent,Soil carbon,Human Induced Regeneration,Avoided deforestation,Energy efficiency,Environmental plantings,Alternative waste treatment,Transport - Land and sea,Savanna fire management (A),Beef cattle herd management,...,Blue Carbon,Carbon capture and storage,Native Forest from Managed Regrowth,Manure management,Landfill gas,Fugitives - Coal,Savanna fire management (S+A),Reforestation and afforestation,Verified Carbon Standard,Wastewater
0,Aboriginal Carbon foundation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,AgCarbon 2020 Pty Ltd,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [74]:
'''mp = (df.groupby([COL_METHOD, "Agent_key"], as_index=False)[COL_VAL]
        .sum()
        .sort_values([COL_METHOD, COL_VAL], ascending=[True, False]))

tables_top10_by_method = {}
for m, sub in mp.groupby(COL_METHOD):
    tbl = (sub.head(15)
             .assign(**{COL_AGENT: sub["Agent_key"].map(display_map)})
             .set_index(COL_AGENT)[[COL_VAL]]
             .rename(columns={COL_VAL: str(m)}))     
    tables_top10_by_method[m] = tbl
    
    
mp_delta_agent = (df.groupby([COL_METHOD, "Agent_key"], as_index=False)[COL_VAL_DELTA]
        .sum()
        .sort_values([COL_METHOD, COL_VAL_DELTA], ascending=[True, False]))

tables_top10_by_method_delta_agent = {}
for m, sub in mp_delta_agent.groupby(COL_METHOD):
    tbl = (sub.head(15)
             .assign(**{COL_AGENT: sub["Agent_key"].map(display_map)})
             .set_index(COL_AGENT)[[COL_VAL_DELTA]]
             .rename(columns={COL_VAL_DELTA: str(m)}))     
    tables_top10_by_method_delta_agent[m] = tbl'''

'mp = (df.groupby([COL_METHOD, "Agent_key"], as_index=False)[COL_VAL]\n        .sum()\n        .sort_values([COL_METHOD, COL_VAL], ascending=[True, False]))\n\ntables_top10_by_method = {}\nfor m, sub in mp.groupby(COL_METHOD):\n    tbl = (sub.head(15)\n             .assign(**{COL_AGENT: sub["Agent_key"].map(display_map)})\n             .set_index(COL_AGENT)[[COL_VAL]]\n             .rename(columns={COL_VAL: str(m)}))     \n    tables_top10_by_method[m] = tbl\n    \n    \nmp_delta_agent = (df.groupby([COL_METHOD, "Agent_key"], as_index=False)[COL_VAL_DELTA]\n        .sum()\n        .sort_values([COL_METHOD, COL_VAL_DELTA], ascending=[True, False]))\n\ntables_top10_by_method_delta_agent = {}\nfor m, sub in mp_delta_agent.groupby(COL_METHOD):\n    tbl = (sub.head(15)\n             .assign(**{COL_AGENT: sub["Agent_key"].map(display_map)})\n             .set_index(COL_AGENT)[[COL_VAL_DELTA]]\n             .rename(columns={COL_VAL_DELTA: str(m)}))     \n    tables_top10_by_method_delta_agent

In [75]:
mp = (df.groupby([COL_METHOD, "Agent_key"], as_index=False)[COL_VAL]
        .sum()
        .sort_values([COL_METHOD, COL_VAL], ascending=[True, False]))

tables_top10_by_method = {}
for m, sub in mp.groupby(COL_METHOD):

    sub2 = sub.copy()
    sub2[COL_AGENT] = sub2["Agent_key"].map(display_map).astype("string").fillna("")
    mask_unknown = (
        sub2[COL_AGENT].str.strip().eq("") |
        sub2[COL_AGENT].str.strip().str.casefold().eq("unknown")
    )
    sub2 = sub2[~mask_unknown]

    tbl = (sub2.head(15)
             .set_index(COL_AGENT)[[COL_VAL]]
             .rename(columns={COL_VAL: str(m)}))

    tables_top10_by_method[m] = tbl

mp_delta_agent = (df.groupby([COL_METHOD, "Agent_key"], as_index=False)[COL_VAL_DELTA]
        .sum()
        .sort_values([COL_METHOD, COL_VAL_DELTA], ascending=[True, False]))

tables_top10_by_method_delta_agent = {}
for m, sub in mp_delta_agent.groupby(COL_METHOD):

    sub2 = sub.copy()
    sub2[COL_AGENT] = sub2["Agent_key"].map(display_map).astype("string").fillna("")

    mask_unknown = (
        sub2[COL_AGENT].str.strip().eq("") |
        sub2[COL_AGENT].str.strip().str.casefold().eq("unknown")
    )
    sub2 = sub2[~mask_unknown]

    tbl = (sub2.head(15)
             .set_index(COL_AGENT)[[COL_VAL_DELTA]]
             .rename(columns={COL_VAL_DELTA: str(m)}))

    tables_top10_by_method_delta_agent[m] = tbl


In [77]:
values = [top10.columns.tolist()] + top10.astype(object).where(top10.notna(), "").values.tolist()
vals3 = [top10_delta_agent.columns.tolist()] + top10_delta_agent.astype(object).where(top10_delta_agent.notna(), "").values.tolist()
ws = get_or_create_ws(sh, "Agent as at latest reg")
write_values_to_ws(ws, values)

ws3 = get_or_create_ws(sh, "Agent delta jun25-latest reg")
write_values_to_ws(ws3, vals3)

ws2 = get_or_create_ws(sh, "Agent by method as at latest reg")
vals2 = tables_dict_to_values(tables_top10_by_method, index_name="Agent")
write_values_to_ws(ws2, vals2)
format_all_numeric_block_except_first_col(ws2, len(vals2), max(len(r) for r in vals2), start_row=1)

ws4 = get_or_create_ws(sh, "Agent by method delta jun25-latest reg")
vals4 = tables_dict_to_values(tables_top10_by_method_delta_agent, index_name="Agent")
write_values_to_ws(ws4, vals4)
format_all_numeric_block_except_first_col(ws4, len(vals4), max(len(r) for r in vals4), start_row=1)

C:\Users\Pawandeep Kaur\AppData\Local\Temp\ipykernel_3144\3924750880.py:66: DeprecationWarning: [Deprecated][in version 6.0.0]: Method signature's arguments 'range_name' and 'values' will change their order. We recommend using named arguments for minimal impact. In addition, the argument 'values' will be mandatory of type: 'List[List]'. (ex) Worksheet.update(values = [[]], range_name=) 
  ws.update("A1", values, value_input_option="USER_ENTERED")


#### erf and cac merged on contract id

In [78]:
ERF_CONTRACT_COL = "Contract ID"   
CAC_CONTRACT_COL = "Carbon Abatement Contract ID"      

def split_cac_ids(x):
    if pd.isna(x):
        return []
    s = str(x).replace("\r", "\n")
    ids = re.findall(r"CAC\d+", s)
    return [i.strip() for i in ids if i.strip()]

erf_expanded = (
    erf_latest
      .assign(**{ERF_CONTRACT_COL: erf_latest[ERF_CONTRACT_COL].apply(split_cac_ids)})
      .explode(ERF_CONTRACT_COL)
)

erf_expanded[ERF_CONTRACT_COL] = erf_expanded[ERF_CONTRACT_COL].astype(str).str.strip()

erf_expanded.head(2)

,Project Proponent,Agent,"Significantly Involved Person(s), Description of involvement",Project Name,Project ID,Method,Method URL,Method Type,Estimation or measurement approach or model,Project Description,...,NKACCUs Units Issued in Financial Year 2023/24,NKACCUs Units Issued in Financial Year 2024/25,NKACCUs Units Issued in Financial Year 2025/26,Name of person/s to whom the NKACCUs issued,Total Number of NKACCUs units relinquished,Enforceable undertaking URLs related to this project,Enforceable undertakings related to this project,Notes,Project revoked,Method Group
0,AGRIPROVE SOLUTIONS CO NO.2 PTY LTD,Agriprove Pty Ltd,NaN,Yona SOC project,ERF191531,Carbon Credits (Carbon Farming Initiative-Esti...,https://www.legislation.gov.au/Details/F2021L0...,Agriculture,NaN,This project increases carbon in soil in the a...,...,0,0,0,NaN,0,NaN,NaN,NaN,No,Soil carbon
1,Glenory Nominees Pty Ltd as Trustee for C J Bu...,Select Carbon Pty Ltd,NaN,New Horizon Soil Carbon Project,ERF204747,Carbon Credits (Carbon Farming Initiative-Esti...,https://www.legislation.gov.au/Details/F2021L0...,Agriculture,NaN,This project increases carbon in soil in the a...,...,0,0,0,NaN,0,NaN,NaN,NaN,No,Soil carbon


In [79]:
cac_level = erf_expanded.merge(
    cac_transformed,                 
    left_on=ERF_CONTRACT_COL,
    right_on=CAC_CONTRACT_COL,
    how="left",
    suffixes=("", "_cac")
)

cac_level.head(2)

,Project Proponent,Agent,"Significantly Involved Person(s), Description of involvement",Project Name,Project ID,Method,Method URL,Method Type,Estimation or measurement approach or model,Project Description,...,Auction date,Volume of abatement sold to the Commonwealth under contract (latest register),Volume of abatement sold to the Commonwealth under contract (june register),CAC deliveries from 1 jan 2025,CAC deliveries from 1 jan 2025 as at june25,Lapsed volume,Outstanding contract volume as at 31-12-2024,Min Delivery Criteria as at latest register,Min Delivery Criteria as at june register,Min Delivery Delta
0,AGRIPROVE SOLUTIONS CO NO.2 PTY LTD,Agriprove Pty Ltd,NaN,Yona SOC project,ERF191531,Carbon Credits (Carbon Farming Initiative-Esti...,https://www.legislation.gov.au/Details/F2021L0...,Agriculture,NaN,This project increases carbon in soil in the a...,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Glenory Nominees Pty Ltd as Trustee for C J Bu...,Select Carbon Pty Ltd,NaN,New Horizon Soil Carbon Project,ERF204747,Carbon Credits (Carbon Farming Initiative-Esti...,https://www.legislation.gov.au/Details/F2021L0...,Agriculture,NaN,This project increases carbon in soil in the a...,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [80]:
with pd.ExcelWriter('cac_analysis.xlsx', engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    cac_level.to_excel(writer, sheet_name="merged_cac_level", index=False)